# Lab L2.2 · ATM cash-demand forecasting — univariate, multivariate and quantile
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 2 · ⏱ 85 min · **Level 1 · From scratch**

Forecast network and per-ATM cash demand with baselines, Holt-Winters, SARIMAX with exogenous calendar drivers, and a direct 7-day-ahead ML model — then turn it into a P90 loading plan.

Follow **L2.2_Lab_Guide.html** step by step. This notebook gives you only the structure — every line of logic is yours. Keep variable names as suggested in the guide so later steps (and later labs) line up.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load the panel and repair outage gaps

1. Load and sort by `atm_id`, `date`. Add `was_outage` (1 where withdrawal is missing).
2. Interpolate `withdrawal_aed` within each ATM (time-based, both directions).
3. Build `net`: network total per date with the calendar columns; save it to `CKPT/atm_network_daily.parquet`.

In [ ]:
# Your code for step 1


## Step 2 · Univariate baselines on a 4-week horizon

1. Split `net` at 2025-08-31 into `hist` and `fut` (next 28 days).
2. Write `fc_metrics(actual, forecast)` → MAE, MAPE, WAPE.
3. Compute the three baselines and store them in `fc` (a DataFrame indexed like `fut`).

In [ ]:
# Your code for step 2


## Step 3 · Holt-Winters (univariate) and SARIMAX with calendar drivers (multivariate)

1. Fit `ExponentialSmoothing(trend='add', seasonal='mul', seasonal_periods=7)` on `hist` from 2022-01-01; forecast 28 days.
2. Fit `SARIMAX(order=(1,0,1), seasonal_order=(1,1,1,7))` on log withdrawals with exogenous calendar flags; forecast with the future flags.
3. Compare all five methods and plot actual vs the two best.

In [ ]:
# Your code for step 3


## Step 4 · Per-ATM features for a direct 7-day-ahead model

1. On `atm` (grouped by ATM) create `lag_7, lag_14, lag_21, lag_28`, `roll7_lag7` (mean of days t-13…t-7) and `roll28_lag7` — all with `shift(7)` first.
2. Add `dow_regime` = day-name + regime, and `month`.
3. Write an `assert` that proves no feature uses data later than t-7 (e.g. compare `lag_7` with the value 7 rows earlier).
4. Keep rows from 2021-07-01; target `y = log1p(withdrawal_aed)`; log-transform lags too.

In [ ]:
# Your code for step 4


## Step 5 · Linear vs gradient-boosting forecasters with TimeSeriesSplit

1. Build a `ColumnTransformer` (scale lags; one-hot `dow_regime`, `location_type`, `atm_id`, `month`; passthrough calendar flags).
2. Tune Ridge alpha with `GridSearchCV(cv=TimeSeriesSplit(4))`.
3. Fit `HistGradientBoostingRegressor(max_iter=400, learning_rate=0.06)` on the same matrix.
4. On test days that were **not outages**, report WAPE for seasonal naive, Ridge and HGB, overall and by `location_type`.

In [ ]:
# Your code for step 5


## Step 6 · P90 quantile forecasts → the loading plan

1. Fit `HistGradientBoostingRegressor(loss='quantile', quantile=0.9)` on the same pipeline.
2. On non-outage test days compute coverage = share of days with actual ≤ P90, and the average buffer (P90 − actual) in AED.
3. Save `ev` columns (date, atm_id, actual, ridge, hgb, p90) to `CKPT/atm_forecasts_test.parquet`.

In [ ]:
# Your code for step 6
